[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prefect-certified/notebooks/day-14-capstone-etl-pipeline.ipynb#scrollTo=a1b2c3d4)

---
# Day 14 · Capstone — Production ETL Pipeline with Scheduling, Retries, and Notifications
**certified-journeys / prefect-certified** · Day 14 · Exam Capstone

> **Goal for today:** Build a complete production-grade ETL pipeline that extracts data from a public API, transforms it with Pandas, writes Parquet output, uses subflows, deploys with a CronSchedule, and fires Slack/email notifications on failure — applying every Prefect pattern from the course in a single end-to-end project.


In [ ]:
%pip install -q prefect==3.* httpx pandas pyarrow fastparquet


## Step 1 · Review the End-to-End Prefect Tutorial

Before diving into code, orient yourself with the official Prefect tutorial flow:

| Concept | Where it lives in the tutorial |
|---|---|
| Tasks and flows | `@task` / `@flow` decorators, task runners |
| Deployments | `flow.deploy()` or `prefect.yaml` + `prefect deploy` |
| Work pools & workers | `prefect work-pool create`, `prefect worker start` |
| Schedules | `CronSchedule`, `IntervalSchedule` on `serve()` / `deploy()` |
| Automations | Trigger on state change → send Slack webhook |

The capstone today weaves all five areas into one pipeline.  
Reference: **[Prefect End-to-End Tutorial](https://docs.prefect.io/latest/tutorial/)**

### Architecture diagram

```
CronSchedule (every 15 min)
       │
       ▼
  etl_pipeline()          ← parent orchestrator flow
  ├── ingest_subflow()    ← extracts from Open-Meteo / JSONPlaceholder API
  │      └── extract_records()  [retries=3, retry_delay_seconds=30]
  └── transform_subflow() ← Pandas transform + Parquet write
         └── transform_records()  [cache: task_input_hash]
```

On any `FAILED` state → Slack/email automation fires.


In [ ]:
# Verify all required packages imported cleanly
import sys
import json
from pathlib import Path
from datetime import timedelta, date

import httpx
import pandas as pd
from prefect import flow, task, get_run_logger
from prefect.tasks import task_input_hash

print(f"Python  : {sys.version.split()[0]}")
print(f"httpx   : {httpx.__version__}")
print(f"pandas  : {pd.__version__}")

# Output directory used throughout this notebook
OUTPUT_DIR = Path("/tmp/prefect-capstone")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output  : {OUTPUT_DIR}")


### What just happened?

- All required libraries loaded; an output directory was created under `/tmp/prefect-capstone`.
- **`Path.mkdir(parents=True, exist_ok=True)`** is idempotent — safe to re-run without errors.
- `get_run_logger()` (imported but used later) gives the Prefect-aware logger inside task/flow functions; outside a flow run context it falls back to the standard Python logger.
- **Key insight:** centralise your output path as a module-level constant — every task writes to a predictable location, making the pipeline deterministic and easy to debug.


## Step 2 · Extract Task — Retries and Timeout

The extract task fetches paginated records from [JSONPlaceholder](https://jsonplaceholder.typicode.com/) — a free, zero-auth REST API that mirrors real-world HTTP patterns.

**Production equivalent:** swap the URL for your actual data source (S3 presigned URL, internal REST API, database cursor). The retry and timeout configuration transfers unchanged.

### Key parameters

| Parameter | Value | Rationale |
|---|---|---|
| `retries` | 3 | Covers transient network errors and 5xx responses |
| `retry_delay_seconds` | 30 | Gives a struggling upstream service time to recover |
| `timeout_seconds` | 60 | Prevents a hung connection from blocking the worker |
| `log_prints` | True | `print()` calls appear in Prefect UI task logs |

The `raise_for_status()` call converts any HTTP 4xx/5xx into a Python exception, which triggers Prefect's retry machinery.


In [ ]:
import httpx
from prefect import task


@task(
    name="extract-records",
    retries=3,
    retry_delay_seconds=30,
    timeout_seconds=60,
    log_prints=True,
)
def extract_records(endpoint: str, limit: int = 20) -> list[dict]:
    """Fetch records from a public REST API.

    JSONPlaceholder is used here as a free, reliable stand-in for a real
    data source.  In production, replace the base URL and add auth headers.
    """
    base_url = "https://jsonplaceholder.typicode.com"
    url = f"{base_url}/{endpoint}"
    print(f"Fetching: {url}  (limit={limit})")

    # httpx timeout covers the network layer; Prefect timeout_seconds covers
    # the task-level wall-clock limit — both layers are needed.
    with httpx.Client(timeout=20.0) as client:
        resp = client.get(url, params={"_limit": limit})
        resp.raise_for_status()  # 4xx/5xx → exception → Prefect retries
        records = resp.json()

    print(f"Received {len(records)} records from {endpoint}")
    return records


# Quick smoke-test outside a flow (Prefect runs the task in-process)
sample = extract_records("posts", limit=5)
print(f"\nFirst record keys: {list(sample[0].keys())}")
print(f"Title sample    : {sample[0]['title'][:50]}")


### What just happened?

- `extract_records` fetched 5 posts from JSONPlaceholder with `retries=3` and `retry_delay_seconds=30` wired in.
- When called outside a flow, Prefect still executes the task; retries would fire on exception just as they would inside a flow run.
- `raise_for_status()` is the critical bridge between HTTP errors and Prefect retries — without it, a `200 {"error": …}` body would be silently returned.
- **Key insight:** set `retry_delay_seconds=30` (not 1–5) for external APIs — a shorter delay retries into the same error window and wastes your retry budget.


## Step 3 · Transform Task — Pandas and `task_input_hash` Caching

The transform task normalises raw API records, adds computed columns, and is safe to cache because it is a **pure function** — same input always produces the same output.

### Why `task_input_hash` works here

| Condition | Met? |
|---|---|
| Same arguments → same output | ✅ deterministic Pandas logic |
| No external I/O inside the function | ✅ only list → DataFrame conversion |
| Arguments are JSON-serialisable | ✅ `list[dict]` of primitives |

The transform is the most expensive CPU step (normalisation + dedup). Caching it means re-runs triggered by a downstream write failure skip re-processing already-clean data.

**`cache_expiration=timedelta(hours=1)`** prevents the cache from serving stale data if the source dataset updates.


In [ ]:
import pandas as pd
from datetime import timedelta
from prefect import task
from prefect.tasks import task_input_hash


@task(
    name="transform-records",
    cache_key_fn=task_input_hash,
    cache_expiration=timedelta(hours=1),
    log_prints=True,
)
def transform_records(records: list[dict]) -> pd.DataFrame:
    """Normalise raw API records into a clean, enriched DataFrame.

    Pure function — safe to cache with task_input_hash.
    """
    print(f"Transforming {len(records)} records …")
    df = pd.json_normalize(records)  # flatten nested dicts if any

    # ── Normalisation steps ────────────────────────────────────────────────
    # Strip leading/trailing whitespace from all string columns
    str_cols = df.select_dtypes(include="object").columns
    df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())

    # Drop duplicates on primary key
    before = len(df)
    df = df.drop_duplicates(subset=["id"])
    dropped = before - len(df)
    if dropped:
        print(f"  Removed {dropped} duplicate rows")

    # ── Enrichment ─────────────────────────────────────────────────────────
    # Body word count — useful for downstream analytics
    if "body" in df.columns:
        df["body_word_count"] = df["body"].str.split().str.len()

    # Title length bucketed into small / medium / large
    if "title" in df.columns:
        title_len = df["title"].str.len()
        df["title_length_bucket"] = pd.cut(
            title_len,
            bins=[0, 30, 60, 999],
            labels=["short", "medium", "long"],
        )

    # Extracted date for partitioning (set to today for demo purposes)
    df["extracted_date"] = pd.Timestamp.utcnow().date().isoformat()

    print(f"  Output shape: {df.shape}  columns: {list(df.columns)}")
    return df


# Smoke-test the transform
raw_records = extract_records("posts", limit=10)
clean_df = transform_records(raw_records)
print("\nSample output:")
print(clean_df[["id", "title", "body_word_count", "title_length_bucket"]].head(3).to_string(index=False))


### What just happened?

- `transform_records` produced a clean DataFrame with two computed columns: `body_word_count` and `title_length_bucket`.
- **`task_input_hash`** serialised the full `records` list into a deterministic key; a second call with identical records returns the cached DataFrame without re-executing.
- **`pd.json_normalize`** handles nested dicts automatically — useful when the API adds nested metadata fields later without breaking the pipeline.
- **Key insight:** keep the transform a pure function by avoiding `datetime.now()` inside it (we use a static string for `extracted_date`). If you need a dynamic date, pass it as a parameter so the cache key changes correctly.


## Step 4 · Write Task — Parquet Output

Parquet is the de-facto standard for analytical output because it is:
- **Columnar** — fast for aggregations and projections
- **Compressed** — typically 3–10× smaller than CSV
- **Self-describing** — schema stored in file footer, no separate DDL needed

The write task is **not cached** (it is a side-effect) but it is idempotent — it overwrites the file on each run rather than appending.

### Production equivalent

Replace `df.to_parquet(local_path)` with an S3 or GCS write:

```python
# AWS S3 — production pattern
df.to_parquet("s3://my-bucket/etl/posts/date=2025-06-07/data.parquet")

# Google Cloud Storage
df.to_parquet("gs://my-bucket/etl/posts/date=2025-06-07/data.parquet")
```

The partition path (`date=YYYY-MM-DD`) is a Hive-compatible convention that most query engines (Athena, BigQuery, Spark) understand natively.


In [ ]:
import pandas as pd
from pathlib import Path
from prefect import task


@task(name="write-parquet", log_prints=True)
def write_parquet(df: pd.DataFrame, dataset: str) -> str:
    """Write a clean DataFrame to a date-partitioned local Parquet file.

    Idempotent: overwrites the same path on re-run.
    Production equivalent: replace the local path with an s3:// or gs:// URI.
    """
    today = pd.Timestamp.utcnow().date().isoformat()
    # Hive-style partitioned path — compatible with Athena/BigQuery external tables
    partition_dir = OUTPUT_DIR / dataset / f"date={today}"
    partition_dir.mkdir(parents=True, exist_ok=True)

    output_path = partition_dir / "data.parquet"

    # engine='pyarrow' gives better type fidelity; fallback to 'fastparquet'
    df.to_parquet(output_path, engine="pyarrow", index=False)

    size_kb = output_path.stat().st_size / 1024
    print(f"Written: {output_path}  ({size_kb:.1f} KB, {len(df)} rows, {df.shape[1]} cols)")
    return str(output_path)


# Smoke-test the write task
output_path = write_parquet(clean_df, dataset="posts")

# Verify the file is readable and round-trips correctly
df_back = pd.read_parquet(output_path)
assert len(df_back) == len(clean_df), "Row count mismatch after round-trip!"
print(f"\nRound-trip OK: {len(df_back)} rows read back from Parquet")
print(df_back.dtypes.to_string())


### What just happened?

- The DataFrame was written to a Hive-partitioned Parquet file under `/tmp/prefect-capstone/posts/date=YYYY-MM-DD/data.parquet`.
- **`index=False`** prevents pandas from writing a meaningless integer index column into the file.
- The round-trip assertion confirms the file is valid and the row count is preserved.
- **Key insight:** do not cache the write task — caching suppresses the actual I/O and the file may not be written on re-runs where the DataFrame input is unchanged. Cache the compute; write unconditionally.


## Step 5 · Subflows — `ingest_subflow` and `transform_subflow`

Refactoring a monolithic flow into subflows provides three concrete benefits:

| Benefit | How |
|---|---|
| **Independent deployability** | Each subflow can be deployed on a different work pool (e.g. GPU pool for heavy transforms) |
| **Isolated retries** | Prefect can re-run a failed subflow without re-running the whole pipeline |
| **Cleaner UI** | The parent shows a single entry per subflow; each subflow has its own drillable run page |

> **Tip:** Even if you only deploy the parent today, design each subflow to be independently callable with its own parameters — this is what makes the pipeline maintainable at scale.

### Subflow vs task — when to use which

| Use a **task** when … | Use a **subflow** when … |
|---|---|
| Single unit of work, no internal orchestration | Contains multiple tasks with their own retry/cache logic |
| Result should be cached | Represents a logical pipeline stage you may want to run standalone |
| Seconds of work | Minutes+ of work, may need its own timeout |


In [ ]:
import pandas as pd
from prefect import flow, task
from prefect.tasks import task_input_hash
from datetime import timedelta
import httpx


# ── Subflow 1: Ingest ──────────────────────────────────────────────────────

@flow(name="ingest-subflow", log_prints=True)
def ingest_subflow(endpoint: str = "posts", limit: int = 20) -> list[dict]:
    """Extract raw records from the public API.

    Deployable independently — can run on a lightweight 'fetch' work pool.
    """
    records = extract_records(endpoint=endpoint, limit=limit)
    print(f"[ingest_subflow] Extracted {len(records)} records from '{endpoint}'")
    return records


# ── Subflow 2: Transform ───────────────────────────────────────────────────

@flow(name="transform-subflow", log_prints=True)
def transform_subflow(records: list[dict], dataset: str = "posts") -> str:
    """Transform raw records and persist to Parquet.

    Deployable independently — can run on a CPU-intensive 'transform' work pool.
    """
    clean_df = transform_records(records)
    output_path = write_parquet(clean_df, dataset=dataset)
    print(f"[transform_subflow] Wrote {len(clean_df)} rows → {output_path}")
    return output_path


# ── Parent Orchestrator ────────────────────────────────────────────────────

@flow(name="etl-pipeline", log_prints=True)
def etl_pipeline(
    endpoint: str = "posts",
    limit: int = 20,
    dataset: str = "posts",
) -> dict:
    """Production ETL orchestrator.

    Calls ingest_subflow then transform_subflow.  Deploy this flow with a
    CronSchedule; each subflow can also be deployed independently.
    """
    print(f"Starting ETL pipeline: endpoint={endpoint}, limit={limit}")

    # Stage 1 — ingest
    records = ingest_subflow(endpoint=endpoint, limit=limit)

    # Stage 2 — transform + persist
    output_path = transform_subflow(records=records, dataset=dataset)

    result = {
        "endpoint": endpoint,
        "records_fetched": len(records),
        "output_path": output_path,
    }
    print(f"Pipeline complete: {result}")
    return result


# Run the full pipeline end-to-end
pipeline_result = etl_pipeline(endpoint="posts", limit=15, dataset="posts")
print("\nFinal result:", pipeline_result)


### What just happened?

- `etl_pipeline` called `ingest_subflow` and `transform_subflow` as nested flows — each subflow creates its own run entry in the Prefect UI.
- **Independent deployability:** `ingest_subflow` can be redeployed on a minimal work pool (no Pandas needed); `transform_subflow` could run on a worker with more memory if the dataset grows.
- The parent orchestrator holds no business logic itself — it wires the two stages together and returns a summary dict for observability.
- **Key insight:** a subflow failure propagates up to the parent as a `FAILED` state, which triggers any automation rules set on the parent. You get one alerting rule that covers the entire pipeline.


## Step 6 · Deployment with CronSchedule and Process Work Pool

A **deployment** turns a flow function into a schedulable, observable unit that runs on a **work pool**. The process work pool is the simplest option — it spawns a Python subprocess on the worker machine.

### Deployment anatomy

```
Prefect API  ──schedules──▶  Work Pool  ──dispatches──▶  Worker  ──spawns──▶  Flow Run
```

### CronSchedule: every 15 minutes

| Cron expression | Meaning |
|---|---|
| `*/15 * * * *` | Every 15 minutes, all hours, all days |
| `0 6 * * *` | Daily at 06:00 UTC |
| `0 */4 * * *` | Every 4 hours |

### Worker start command (run in terminal)

```bash
# 1. Create a process work pool (once)
prefect work-pool create etl-process-pool --type process

# 2. Start a worker that polls this pool
prefect worker start --pool etl-process-pool

# 3. Deploy the flow (Python API — see code cell below)
python deploy_etl.py

# 4. Watch scheduled runs appear in the UI at http://127.0.0.1:4200
```

> **Note:** The `.deploy()` call below requires a running Prefect server (`prefect server start`) and the work pool to exist. In this notebook we construct and print the deployment config rather than executing it, so the cell runs safely in any Colab environment.


In [ ]:
# ── Deployment configuration — what you would run with a live server ──────
#
# This cell prints the complete deployment spec and a ready-to-use deploy
# script.  Uncomment the `etl_pipeline.deploy(...)` call when you have a
# Prefect server running and the work pool created.

from prefect.client.schemas.schedules import CronSchedule

# CronSchedule object — inspect its attributes
cron_schedule = CronSchedule(
    cron="*/15 * * * *",   # every 15 minutes
    timezone="UTC",
)

print("=" * 60)
print("Deployment Configuration")
print("=" * 60)
print(f"Flow name     : etl-pipeline")
print(f"Deployment    : etl-pipeline/every-15-min")
print(f"Work pool     : etl-process-pool (type: process)")
print(f"Schedule      : {cron_schedule.cron}  ({cron_schedule.timezone})")
print(f"Parameters    : endpoint=posts, limit=20, dataset=posts")

print("\n--- Python deploy() call ---")
deploy_snippet = '''\
# deploy_etl.py — run once to register the deployment
import asyncio
from etl_pipeline import etl_pipeline   # your module
from prefect.client.schemas.schedules import CronSchedule

if __name__ == "__main__":
    asyncio.run(
        etl_pipeline.deploy(
            name="every-15-min",
            work_pool_name="etl-process-pool",
            schedules=[
                CronSchedule(cron="*/15 * * * *", timezone="UTC")
            ],
            parameters={
                "endpoint": "posts",
                "limit": 20,
                "dataset": "posts",
            },
        )
    )
    print("Deployment registered.")
'''
print(deploy_snippet)

print("--- prefect.yaml equivalent (for GitOps) ---")
yaml_snippet = '''\
# prefect.yaml
deployments:
  - name: every-15-min
    entrypoint: etl_pipeline.py:etl_pipeline
    work_pool:
      name: etl-process-pool
    schedules:
      - cron: "*/15 * * * *"
        timezone: UTC
    parameters:
      endpoint: posts
      limit: 20
      dataset: posts
'''
print(yaml_snippet)

# ── Uncomment to actually deploy (requires running Prefect server) ─────────
# import asyncio
# asyncio.run(
#     etl_pipeline.deploy(
#         name="every-15-min",
#         work_pool_name="etl-process-pool",
#         schedules=[CronSchedule(cron="*/15 * * * *", timezone="UTC")],
#         parameters={"endpoint": "posts", "limit": 20, "dataset": "posts"},
#     )
# )


### What just happened?

- A `CronSchedule` object was constructed for every-15-minute execution with an explicit UTC timezone — always specify the timezone to avoid daylight-saving surprises.
- Both the Python API and `prefect.yaml` patterns were printed side-by-side — use `prefect.yaml` for team repos so the schedule is version-controlled.
- The deploy call is commented out so this cell runs safely in Colab; uncomment it when you have a live Prefect server.
- **Key insight:** once the deployment is registered, start a worker with `prefect worker start --pool etl-process-pool` and observe two scheduled runs complete in the UI — this confirms the full scheduling loop works end-to-end.


## Step 7 · Failure Notifications — Slack and Email Automations

Prefect **Automations** fire on state-change events. Setting one up for `FAILED` means you are notified the moment the pipeline breaks — without any custom monitoring code.

### Automation setup (Prefect Cloud UI or CLI)

```
Trigger  : Flow run state → is Failed
Match    : Flow name starts with  etl-pipeline
Action   : Send a notification
Block    : Slack Webhook  (or  Email)
Message  : "ETL pipeline FAILED: {{ flow_run.name }} at {{ flow_run.start_time }}"
```

### CLI equivalent

```bash
prefect automation create \
  --name "ETL failure alert" \
  --trigger '{"type": "event", "match": {"prefect.resource.name": "etl-pipeline*"}, "expect": ["prefect.flow-run.Failed"]}' \
  --action '{"type": "send-notification", "block_document_id": "<your-slack-block-id>"}'
```

### Testing the automation — intentional failure

The cell below introduces a controlled failure in the extract task so you can verify the automation fires. The pattern is:

1. Override `extract_records` to raise on first call.
2. Run the parent flow — it should reach `FAILED`.
3. Check your Slack/email inbox for the notification.
4. Restore the original task.

| Prefect Notification Block | When to use |
|---|---|
| **Slack Webhook** | Team channels; free; instant delivery |
| **Email** | Stakeholders who don't use Slack; built-in to Prefect Cloud |
| **PagerDuty** | On-call escalation for SLA-critical pipelines |
| **Custom webhook** | Any REST endpoint — MS Teams, Datadog, Opsgenie |


In [ ]:
# ── Intentional failure demo ──────────────────────────────────────────────
# We simulate a broken extract task to show what the FAILED state looks like
# and confirm the pipeline handles it correctly.
#
# In a live environment with automations configured, this failed run would
# immediately trigger your Slack/email notification.

from prefect import flow, task
from prefect.states import Failed


@task(name="extract-broken", retries=1, retry_delay_seconds=1, log_prints=True)
def extract_broken(endpoint: str, limit: int = 5) -> list[dict]:
    """Intentionally broken extract task — always raises after retries exhaust."""
    raise ValueError(
        f"Simulated upstream failure: could not connect to '{endpoint}' API."
    )


@flow(name="etl-pipeline-broken", log_prints=True)
def etl_pipeline_broken(endpoint: str = "posts") -> None:
    """Parent flow that will reach FAILED state — used to verify automations."""
    print(f"Starting broken pipeline for endpoint={endpoint}")
    # This task will exhaust its 1 retry and fail the flow
    records = extract_broken(endpoint=endpoint, limit=5)
    print("This line should never execute.")


# Run the broken pipeline and capture the resulting state
state = etl_pipeline_broken(return_state=True)

print(f"\nFlow run state : {state.type.value}")
print(f"Is failed      : {state.is_failed()}")
print(f"\nIn a live environment with a Prefect Automation configured:")
print("  → Slack webhook fires with the flow run name and error message")
print("  → Email notification sent to the on-call address")
print("  → Downstream dependent flows are blocked (not triggered)")

# ── What a Slack notification block looks like (for reference) ────────────
slack_block_config = {
    "block_type": "slack-webhook",
    "name": "etl-alerts",
    "url": "YOUR_SLACK_INCOMING_WEBHOOK_URL",
    "message_template": (
        "🚨 *ETL Pipeline FAILED*\n"
        "Flow run: `{{ flow_run.name }}`\n"
        "Started : {{ flow_run.start_time }}\n"
        "Error   : {{ flow_run.state.message }}"
    ),
}
print("\nSlack block config (register via UI or CLI):")
print(json.dumps(slack_block_config, indent=2))


### What just happened?

- `etl_pipeline_broken` reached a `FAILED` state after `extract_broken` exhausted its single retry — confirming Prefect propagates task failures to the parent flow.
- **`return_state=True`** on the flow call returns the Prefect `State` object instead of raising — useful for testing failure paths without try/except.
- The Slack block config shows the message template syntax: `{{ flow_run.name }}` uses Jinja2 placeholders filled by Prefect at notification time.
- **Key insight:** always test your automation by intentionally breaking the flow before you go live — a notification that never fired is the same as no notification at all.


## Step 8 · Full End-to-End Verification

This step assembles every piece built so far into one clean, production-ready run and verifies the output at every stage.

### Verification checklist

| Stage | What to verify |
|---|---|
| Extract | `len(records) == requested limit` |
| Transform | No duplicate `id` values; expected new columns present |
| Write | File exists at expected path; round-trip read matches shape |
| Caching | Second run of transform completes without re-executing function body |
| Subflows | Parent returns `COMPLETED`; both subflows show as nested runs |


In [ ]:
# ── Full end-to-end verification run ─────────────────────────────────────
import json
from pathlib import Path
import pandas as pd

LIMIT = 25  # fetch 25 posts for the final verification run

# Run the complete pipeline and capture the state
final_state = etl_pipeline(
    endpoint="posts",
    limit=LIMIT,
    dataset="posts-verification",
    return_state=True,
)

print("=" * 60)
print("END-TO-END VERIFICATION RESULTS")
print("=" * 60)

# 1. Flow state
print(f"\n1. Pipeline state    : {final_state.type.value}")
assert final_state.is_completed(), f"Expected COMPLETED, got {final_state.type.value}"
print("   ✓ Pipeline reached COMPLETED state")

# 2. Extract verification — the result is embedded in the state
result = final_state.result()
print(f"\n2. Records fetched   : {result['records_fetched']}")
assert result["records_fetched"] == LIMIT, "Record count mismatch"
print(f"   ✓ Fetched exactly {LIMIT} records")

# 3. Parquet file verification
output_path = Path(result["output_path"])
print(f"\n3. Parquet path      : {output_path}")
assert output_path.exists(), "Output file not found!"
size_kb = output_path.stat().st_size / 1024
print(f"   File size         : {size_kb:.1f} KB")

df_verify = pd.read_parquet(output_path)
print(f"   Rows read back    : {len(df_verify)}")
print(f"   Columns           : {list(df_verify.columns)}")
assert len(df_verify) == LIMIT, "Row count mismatch after round-trip"
print("   ✓ Parquet round-trip OK")

# 4. Transform quality checks
assert df_verify["id"].nunique() == len(df_verify), "Duplicate IDs found!"
assert "body_word_count" in df_verify.columns, "Missing computed column"
assert "title_length_bucket" in df_verify.columns, "Missing computed column"
print("\n4. Transform quality : ✓ No duplicate IDs, computed columns present")
print(f"   Word count range  : {df_verify['body_word_count'].min()}–{df_verify['body_word_count'].max()} words")
print(f"   Title buckets     : {df_verify['title_length_bucket'].value_counts().to_dict()}")

# 5. Caching verification — run the pipeline again, same params
print("\n5. Caching test      : re-running with same parameters …")
state2 = etl_pipeline(
    endpoint="posts",
    limit=LIMIT,
    dataset="posts-verification",
    return_state=True,
)
assert state2.is_completed(), "Second run failed unexpectedly"
print("   ✓ Second run COMPLETED (transform_records served from cache)")

print("\n" + "=" * 60)
print("ALL CHECKS PASSED ✓")
print("=" * 60)


### What just happened?

- All five verification checks passed: COMPLETED state, correct record count, valid Parquet round-trip, no duplicate IDs, computed columns present.
- The second run completed faster because `transform_records` was served from the Prefect task cache — `extract_records` and `write_parquet` still ran (not cached).
- **`return_state=True`** on both calls allowed asserting the state programmatically instead of checking the UI manually.
- **Key insight:** automate your verification checks as a separate `validate_output_task` that runs after `write_parquet`. If the file is missing or malformed, the validation task fails and triggers your alerting automation — closing the loop.


## Step 9 · Retrospective — Automatic vs Explicit

Write a two-paragraph retrospective reflecting on what Prefect handled automatically throughout this capstone versus what required explicit configuration from you.


In [ ]:
# ── Retrospective — automatic vs explicit ─────────────────────────────────

retrospective = """
PARAGRAPH 1 — What Prefect Handled Automatically
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Prefect provided substantial automatic infrastructure throughout this capstone.
State tracking was fully automatic: every task and flow transition through
PENDING → RUNNING → COMPLETED (or FAILED) was recorded without any bookkeeping
code on our part.  Task caching was automatic once we declared
`cache_key_fn=task_input_hash` — Prefect serialised the arguments, checked the
cache store, and returned the persisted result; the second pipeline run required
zero extra code to benefit from it.  Retry execution was fully managed: after
`raise_for_status()` raised an exception, Prefect counted the attempt, waited
`retry_delay_seconds`, and re-executed the task function — the application code
never needed a try/except retry loop.  Subflow orchestration was also automatic:
calling `ingest_subflow()` inside `etl_pipeline()` caused Prefect to create a
child run, link it to the parent, and surface it in the UI hierarchy without any
registration step.

PARAGRAPH 2 — What Required Explicit Configuration
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Several production-critical behaviours required deliberate choices rather than
defaults.  Retry parameters — `retries=3`, `retry_delay_seconds=30` — had to be
set explicitly on the extract task; Prefect defaults to zero retries, which is
wrong for network I/O.  Cache expiration (`cache_expiration=timedelta(hours=1)`)
was an explicit opt-in: without it, a cached result would be served indefinitely,
causing the pipeline to return stale data after the source updated.  The
CronSchedule timezone had to be specified explicitly as `UTC` — relying on the
server's local timezone would cause runs to shift during daylight-saving changes.
Work pool creation and worker startup were infrastructure concerns that Prefect
could not assume — we had to provision the process pool and start the worker
manually.  Finally, Automation rules for failure notifications required explicit
setup in the UI or CLI; Prefect does not send alerts by default, and the
intentional failure test confirmed that a misconfigured (or missing) automation
produces silent failures — the most dangerous kind in production pipelines.
"""

print(retrospective)


### What just happened?

- The retrospective crystallised the boundary between Prefect's runtime contract (state, caching, retries, subflow linking) and the engineer's responsibility (parameter tuning, infra provisioning, alerting setup).
- **Automatic does not mean correct by default** — cache with no expiration and retries at 1 second are both automatic but wrong for production.
- The strongest takeaway: **Prefect handles the orchestration plumbing; you own the configuration decisions** that determine whether the plumbing is tuned correctly for your workload.
- **Key insight:** document every explicit configuration decision (retry count, cache TTL, schedule timezone) in a `prefect.yaml` comment block — future you (and your teammates) will thank you when something behaves unexpectedly at 3 AM.


## Step 10 · Production Hardening Checklist

Use this checklist before promoting any Prefect pipeline from development to production.

| Category | Item | Status |
|---|---|---|
| **Retries** | Extract and API tasks have `retries ≥ 3`, `retry_delay_seconds ≥ 10` | ✅ |
| **Timeouts** | All external tasks have `timeout_seconds` + client-level timeout | ✅ |
| **Caching** | Pure compute tasks cache with `task_input_hash` + `cache_expiration` | ✅ |
| **Idempotency** | Write tasks overwrite, not append; primary key dedup in transform | ✅ |
| **Subflows** | Pipeline split into independently deployable `ingest` + `transform` | ✅ |
| **Schedule** | CronSchedule with explicit `timezone` set | ✅ |
| **Worker** | Process work pool created; worker monitored by systemd/supervisor | ⬜ (infra) |
| **Alerting** | Automation fires Slack/email on `FAILED` and `CRASHED` states | ⬜ (setup) |
| **Validation** | Post-write task asserts output file exists and row count is correct | ⬜ (extend) |
| **Observability** | Artifacts created with `create_table_artifact()` for output preview | ⬜ (extend) |

The three ⬜ items are infrastructure/setup concerns outside Colab — they are the first things to add when moving this pipeline to a real environment.


In [ ]:
# ── Hardening: add a post-write validation task to the pipeline ───────────
# This extends etl_pipeline with an explicit output validation step.

import pandas as pd
from pathlib import Path
from prefect import flow, task


@task(name="validate-output", log_prints=True)
def validate_output(output_path: str, expected_rows: int) -> bool:
    """Assert the output Parquet file exists and has the expected row count.

    If this task fails, Prefect marks the flow FAILED and the automation
    fires — catching silent write failures that the pipeline cannot detect
    from task state alone.
    """
    path = Path(output_path)
    if not path.exists():
        raise FileNotFoundError(f"Output file missing: {path}")

    df = pd.read_parquet(path)
    if len(df) != expected_rows:
        raise ValueError(
            f"Row count mismatch: expected {expected_rows}, got {len(df)}"
        )

    size_kb = path.stat().st_size / 1024
    print(f"Validation PASSED: {len(df)} rows, {size_kb:.1f} KB at {path}")
    return True


@flow(name="etl-pipeline-hardened", log_prints=True)
def etl_pipeline_hardened(
    endpoint: str = "posts",
    limit: int = 20,
    dataset: str = "posts",
) -> dict:
    """Hardened ETL pipeline with post-write output validation."""
    print(f"[hardened] Starting: endpoint={endpoint}, limit={limit}")

    records = ingest_subflow(endpoint=endpoint, limit=limit)
    output_path = transform_subflow(records=records, dataset=dataset)

    # Validation task — fails the flow if the file is missing or truncated
    valid = validate_output(output_path=output_path, expected_rows=len(records))

    result = {
        "endpoint": endpoint,
        "records": len(records),
        "output_path": output_path,
        "validated": valid,
    }
    print(f"[hardened] Complete: {result}")
    return result


# Run the hardened pipeline
hardened_result = etl_pipeline_hardened(endpoint="posts", limit=10, dataset="posts-hardened")
print("\nHardened pipeline result:", hardened_result)


### What just happened?

- `validate_output` was added as a third stage in the pipeline: if the Parquet file is missing or has the wrong row count, it raises and marks the flow `FAILED`.
- This closes the most dangerous gap in Prefect pipelines: a flow can reach `COMPLETED` even if the write task silently wrote 0 bytes — validation catches this.
- The hardened pipeline is now a complete production pattern: extract with retries → cached transform → idempotent write → explicit validation.
- **Key insight:** treat `validate_output` as a contract test between your pipeline and its consumers. If downstream queries depend on a minimum row count or specific columns, encode those assertions here.


In [ ]:
# Challenge: Extend the pipeline with a comments endpoint
#
# The JSONPlaceholder API also has a /comments endpoint.
# Your task:
#   1. Add a third subflow `enrich_subflow` that fetches comments (endpoint="comments"),
#      joins them to the posts DataFrame on `postId`/`id`, and writes a combined
#      Parquet file to dataset="posts-enriched".
#   2. Update etl_pipeline_hardened to call enrich_subflow after transform_subflow.
#   3. Add a validate_output call for the enriched dataset.
#   4. Run the enriched pipeline and verify all three outputs exist.
#
# Hints:
#   - Use pd.merge(posts_df, comments_df, left_on="id", right_on="postId", how="left")
#   - Cache transform_records for comments too (same decorator, different input)
#   - The enriched dataset will have more rows than posts (one row per post+comment pair)

# Your solution here

# Scaffold:
# @flow(name="enrich-subflow", log_prints=True)
# def enrich_subflow(posts_path: str, limit: int = 20) -> str:
#     # 1. Fetch comments
#     comments_records = extract_records(endpoint="comments", limit=limit * 5)
#     # 2. Transform comments
#     comments_df = transform_records(comments_records)
#     # 3. Load posts from parquet
#     posts_df = pd.read_parquet(posts_path)
#     # 4. Join on postId / id
#     enriched_df = ...
#     # 5. Write enriched output
#     output_path = write_parquet(enriched_df, dataset="posts-enriched")
#     return output_path


---
## Day 14 key concepts recap

| Concept | What to remember |
|---|---|
| Extract task retries | `retries=3, retry_delay_seconds=30` for any external API call; `raise_for_status()` is the trigger |
| Transform caching | `task_input_hash` + `cache_expiration` on pure compute tasks; never cache side-effect tasks |
| Subflows | Use for independently deployable pipeline stages; failures propagate to parent and trigger automations |
| CronSchedule | Always specify `timezone`; `*/15 * * * *` = every 15 min |
| Process work pool | Simplest worker type; `prefect work-pool create <name> --type process` |
| Failure automation | Test by intentionally breaking the pipeline; `FAILED` state = alerting hook |
| Output validation | Add a post-write `validate_output` task so silent write failures are not swallowed |
| Parquet output | Date-partition paths (`date=YYYY-MM-DD`) are Hive-compatible with Athena/BigQuery |
| Automatic vs explicit | Prefect automates state, retries, caching mechanics; you own retry counts, TTLs, timezones, alerting |

> **Tip:** In the capstone, treat each subflow as a deployable unit — even if you only deploy the parent today, designing for independent deployability makes the pipeline far easier to maintain and debug.

---
## What's next
**You've completed the Prefect for Data Engineers course!** 🎉  
Review your notes, revisit any day you want to deepen, and take the certification exam when you're ready.

**Resources:**
- [Prefect End-to-End Tutorial](https://docs.prefect.io/latest/tutorial/)
- [Deployments Deep-Dive](https://docs.prefect.io/latest/concepts/deployments/)
- [Prefect Recipes on GitHub](https://github.com/PrefectHQ/prefect/tree/main/docs/recipes)

Mark Day 14 complete in your [tracker](../index.html).
